# TriCompile
Author: Swayam Shakya

Run from the project folder. This notebook imports the production compiler, demonstrates its pipeline, and starts the web interface. Python 3.10+ is required; Jupyter is only needed for this notebook.

In [1]:
from pathlib import Path
import sys
assert Path("core.py").exists(), "Open the notebook from the TriCompile project folder"
from core import compile_source, encode, decode, Machine
from runners import capabilities, run_native
print(capabilities())

{'python': True, 'javascript': True, 'c': True, 'cpp': True}


## Compile and run beginner commands

In [2]:
source = "set total to 6 * 7\nshow total"
instructions = compile_source(source)
for index, instruction in enumerate(instructions):
    print(index, instruction)
assert Machine(instructions).run() == "42"

0 {'op': 'PUSH', 'arg': 6, 'line': 1}
1 {'op': 'PUSH', 'arg': 7, 'line': 1}
2 {'op': 'MUL', 'arg': None, 'line': 1}
3 {'op': 'STORE', 'arg': 'total', 'line': 1}
4 {'op': 'LOAD', 'arg': 'total', 'line': 2}
5 {'op': 'PRINT', 'arg': None, 'line': 2}
6 {'op': 'HALT', 'arg': None, 'line': 0}


## Inspect real serialized bytecode

In [3]:
bytecode = encode(instructions)
print(bytecode.hex())
assert decode(bytecode) == instructions

54434d50015b7b226f70223a2250555348222c22617267223a362c226c696e65223a317d2c7b226f70223a2250555348222c22617267223a372c226c696e65223a317d2c7b226f70223a224d554c222c22617267223a6e756c6c2c226c696e65223a317d2c7b226f70223a2253544f5245222c22617267223a22746f74616c222c226c696e65223a317d2c7b226f70223a224c4f4144222c22617267223a22746f74616c222c226c696e65223a327d2c7b226f70223a225052494e54222c22617267223a6e756c6c2c226c696e65223a327d2c7b226f70223a2248414c54222c22617267223a6e756c6c2c226c696e65223a307d5d


## Step through execution

In [4]:
machine = Machine(instructions)
while not machine.halted:
    machine.step()
    print(machine.state())

{'pc': 1, 'stack': [6], 'variables': {}, 'output': '', 'steps': 1, 'halted': False}
{'pc': 2, 'stack': [6, 7], 'variables': {}, 'output': '', 'steps': 2, 'halted': False}
{'pc': 3, 'stack': [42], 'variables': {}, 'output': '', 'steps': 3, 'halted': False}
{'pc': 4, 'stack': [], 'variables': {'total': 42}, 'output': '', 'steps': 4, 'halted': False}
{'pc': 5, 'stack': [42], 'variables': {'total': 42}, 'output': '', 'steps': 5, 'halted': False}
{'pc': 6, 'stack': [], 'variables': {'total': 42}, 'output': '42', 'steps': 6, 'halted': False}
{'pc': 7, 'stack': [], 'variables': {'total': 42}, 'output': '42', 'steps': 7, 'halted': True}


## Execute a popular language

In [5]:
result = run_native("python", "print(int(input()) * 2)", "21\n")
print(result)
assert result["stdout"].strip() == "42"

{'stdout': '42\n', 'stderr': '', 'exit_code': 0, 'timed_out': False}


## Start the interface
Run once. Open the printed URL. Use the final cell when finished.

In [6]:
from server import start
app = start(open_browser=False)

TriCompile: http://127.0.0.1:8765


In [7]:
app.shutdown()
app.server_close()